In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [5]:
torch.manual_seed(42)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_head):
        super().__init__()
        assert d_model % num_head == 0
        self.d_model = d_model; self.num_head = num_head; self.d_k = d_model // num_head
        self.W_q = nn.Linear(d_model, d_model); self.W_k = nn.Linear(d_model, d_model); self.W_v = nn.Linear(d_model, d_model); self.W_o = nn.Linear(d_model, d_model)
    def split_head(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.num_head, self.d_k).transpose(1,2)
    def combine_head(self, x):
        B, H, T, d_k = x.shape
        return x.transpose(1,2).contiguous().view(B, T, H*d_k)
    def forward(self, Q, K, V, mask=None):
        Q = self.split_head(self.W_q(Q)); K = self.split_head(self.W_k(K)); V = self.split_head(self.W_v(V))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1)
        output = weights @ V
        output = self.combine_head(output)
        return self.W_o(output), weights

B, T, D, H = 2, 5, 16, 4
x = torch.rand(B, T, D)
mha = MultiHeadAttention(D, H)
out, weights = mha(x,x,x)
print("input:", x.shape)
print("output:", out.shape)
print("attention weights:", weights.shape)

input: torch.Size([2, 5, 16])
output: torch.Size([2, 5, 16])
attention weights: torch.Size([2, 4, 5, 5])
